# 01. Annual harmonic recipe: development-only selection and freezing

Adds a third model family beside the fixed and aligned window models, which stay unchanged as the baseline. Whole-year Sentinel-2 harmonic predictors with Sentinel-1 radar need no four-month window or green-up date. Every choice here uses development data only: spatial CV within the locked training split and holding out one development dataset at a time. Test parcels are read once, in Annual 02.

**Before running:** Preprocessing 01 (the quality handoff these inputs read). The dataset is whatever your current Preprocessing 01 handoff contains, so results follow your own data. Building the inputs samples Sentinel-1 at the selected pixels by small direct Earth Engine requests (`earth_engine_project` in `settings.local.json`; about 260 requests the first time) cached under `pipeline_outputs/annual/cache/sentinel1/`; reruns read the cache. Running this notebook publishes new `current_*` pointers under `pipeline_outputs/annual/`.

Alternatives that were tried and not adopted are in `experiments/` (see its README).

In [ ]:
from pathlib import Path
import json
import sys

root = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (path / "pyproject.toml").is_file())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

from experiments.annual import workflow as annual_workflow

## Annual inputs

All parcels from the verified quality handoff, up to 20 pixels per parcel, with development pixels shared by different parcel groups removed.

In [ ]:
inputs = annual_workflow.build_inputs(root)
record = json.loads(inputs.read_text())
print(inputs.relative_to(root))
print({key: record[key] for key in ("pixels", "parcels", "pixels_per_parcel")})

## Select the recipe on development views

The rule is fixed before scoring: Sentinel-2 only and Sentinel-2 + radar are scored with the default Random Forest; the highest mean of spatial CV and dataset-holdout macro-F1 wins, ties to fewer predictors. The default Random Forest is then frozen.

In [ ]:
selection = annual_workflow.select(root)
chosen = json.loads((selection.parent / "selection.json").read_text())
for name, scores in chosen["candidates"].items():
    print(f"{name:14} predictors {scores['predictors']:3}  locked validation {scores['locked_validation_macro_f1']:.3f}  "
          f"spatial CV {scores['spatial_cv_macro_f1']:.3f}  dataset holdout {scores['dataset_holdout_macro_f1']:.3f}  "
          f"selection {scores['selection_score']:.3f}")
print("\nSelected:", chosen["selected"], "| classifier:", chosen["classifier"])

## Freeze

The selected recipe is fitted with the default Random Forest (1,000 trees) on the locked training split with the main pipeline's weights. Locked validation is reported for reference.

In [ ]:
models = annual_workflow.freeze(root)
info = json.loads((models.parent / "model.json").read_text())
print(models.relative_to(root))
print("Classifier:", info["classifier"]["family"], info["classifier"]["setting"], info["classifier"]["backend"] or "")
print("Locked validation macro-F1:", round(info["validation"]["macro_f1"], 3),
      "| training majority:", info["training_majority_class"])